# Assignment 4 - Simple Neural Networks (**PyTorch Version**)

You are building a fraud classifier from transaction data.


## What stays the same from the original assignment

- **Task:** predict fraud from transaction data.
- **Deliverables:** one training notebook and one prediction notebook.
- **Grading:**
  - Code produces predictions — **40**
  - Accuracy — **30**
  - Explanation — **20**
  - Balance / variable transformations — **10**

## What changes in this version

- `tensorflow/keras` is replaced by **PyTorch**.
- You will use:
  - `TensorDataset` / `DataLoader`
  - `nn.Module`
  - a manual training loop
  - `BCEWithLogitsLoss` for binary classification

## Reference style

The notebook structure below is intentionally influenced by the PyTorch teaching style used in your reference notebook: device selection, tensor-based pipelines, and explicit training loops.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

import joblib
from pathlib import Path

# Device setup in the same spirit as the reference notebook
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


## Load Data

Expected training file: `fraudTrain.csv.zip`


In [ ]:
df = pd.read_csv("fraudTrain.csv.zip")
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print(df.shape)
df.head()


In [ ]:
df.describe(include="all").T


### Deal with Lat/Lon

A plain latitude/longitude pair is not usually ideal for a tabular neural network.
A more useful transformation is to create a **distance-like feature** between the customer and merchant.

**TODO:** create at least one useful location feature.
Suggested idea: distance between `(lat, long)` and `(merch_lat, merch_long)`.


In [ ]:
# TODO: create location-based features
# Suggested: a haversine distance feature in km

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# Example:
# df["distance_km"] = haversine_km(df["lat"], df["long"], df["merch_lat"], df["merch_long"])


### Deal with Time

The original prompt suggests that time and DOB are not especially useful in raw form, but can become useful after transformation.

**TODO:** create time-based features such as:
- hour of day
- day of week
- month
- customer age
- cyclical encoding of hour


In [ ]:
# TODO: create time-based features
# Suggested starting point:

# df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
# df["dob"] = pd.to_datetime(df["dob"])
# df["hour"] = df["trans_date_trans_time"].dt.hour
# df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
# df["month"] = df["trans_date_trans_time"].dt.month
# df["customer_age"] = ((df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25).astype(float)
# df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
# df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)


### Check Target Balance

This dataset is imbalanced, so you should inspect the fraud rate before training.


In [ ]:
# TODO: inspect target balance
# Example:
# df["is_fraud"].value_counts(normalize=True)


### Prepare Data

Recommended workflow:
1. Choose features.
2. Separate numeric and categorical columns.
3. One-hot encode categorical variables.
4. Standardize numeric variables.
5. Combine everything into a single feature matrix.

You should also decide which columns to drop because they are IDs, raw timestamps, or leakage-prone text fields.


In [ ]:
# TODO: build your feature set

TARGET = "is_fraud"

# Suggested engineered columns you may use after creating them:
# engineered_numeric = [
#     "amt", "city_pop", "distance_km", "customer_age",
#     "hour_sin", "hour_cos", "day_of_week", "month"
# ]
# categorical_cols = ["category", "gender"]

# drop_cols = [
#     TARGET,
#     "trans_date_trans_time", "dob",
#     "first", "last", "street", "city", "state", "job",
#     "trans_num", "merchant", "cc_num",
#     "lat", "long", "merch_lat", "merch_long",
#     "unix_time", "zip"
# ]


### Split Data

Use a stratified split so the fraud rate stays similar in train/validation/test.


In [ ]:
# TODO: split your data
# Suggested pattern:
# X_train, X_temp, y_train, y_temp = train_test_split(..., stratify=y, test_size=0.30, random_state=42)
# X_valid, X_test, y_valid, y_test = train_test_split(..., stratify=y_temp, test_size=0.50, random_state=42)


### Convert to Tensors and DataLoaders

Follow the reference notebook style:
- convert arrays to tensors
- wrap them in `TensorDataset`
- use `DataLoader`
- handle imbalance with either:
  - `WeightedRandomSampler`, or
  - `pos_weight` in `BCEWithLogitsLoss`


In [ ]:
# TODO: create tensors / dataloaders
# Example outline:
# X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
# y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
# train_ds = TensorDataset(X_train_tensor, y_train_tensor)
# train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)


### Model

Build a small multilayer perceptron (MLP) for binary classification.

Suggested starting point:
- Linear → ReLU → Dropout
- Linear → ReLU → Dropout
- Linear → 1 output logit


In [ ]:
# TODO: define your PyTorch model

class FraudMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)


### Train the Model

Use:
- `BCEWithLogitsLoss`
- `Adam`
- an epoch loop with training and validation loss

You should move both the model and each batch to `device`.


In [ ]:
# TODO: write training loop
# Track at least:
# - training loss
# - validation loss
# - validation ROC-AUC or another useful metric


### Evaluate

At minimum, produce:
- confusion matrix
- classification report
- ROC-AUC if possible

You may also tune the classification threshold rather than always using 0.50.


In [ ]:
# TODO: evaluate on validation/test data


### Save What the Prediction Notebook Needs

Your prediction notebook should be able to `Run all` without retraining.
So save:
- the trained model weights
- the scaler
- the encoder
- the list of feature columns / metadata needed for preprocessing


In [ ]:
# TODO: save artifacts
# Suggested filenames:
# torch.save(model.state_dict(), "fraud_mlp_state_dict.pt")
# joblib.dump(scaler, "fraud_scaler.joblib")
# joblib.dump(encoder, "fraud_encoder.joblib")


## Short Explanation (write your own)

In your final submission, include a short explanation covering:
- imbalance handling
- treatment of location and time variables
- model structure
- optimization choices such as dropout, regularization, thresholding, or feature selection
